# Patrón de Comportamiento: Chain of Responsibility

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Chain of Responsibility** (cadena de responsabilidad) pasa una solicitud a lo
largo de una **cadena de manejadores**. Cada manejador decide si la **atiende** o la
**pasa** al siguiente. El emisor no necesita saber quién la resolverá.

### ¿Qué problema resuelve en la banca?
La **aprobación de un crédito** depende del monto: un **cajero** aprueba montos pequeños,
un **supervisor** montos medianos, un **gerente** montos altos, y por encima de cierto
límite se **rechaza**. Sin el patrón, terminamos con un `if/elif` gigante que mezcla todos
los niveles y hay que modificar cada vez que cambian los límites o aparece un nivel nuevo.

## Código *sin patrón* (el problema es evidente)
Un único bloque `if/elif` concentra todos los niveles de aprobación.

In [1]:
def aprobar_credito_sin_patron(monto: float) -> str:
    if monto <= 1_000_000:
        return f"Cajero aprueba ${monto}"
    elif monto <= 10_000_000:
        return f"Supervisor aprueba ${monto}"
    elif monto <= 50_000_000:
        return f"Gerente aprueba ${monto}"
    else:
        return f"RECHAZADO: ${monto} excede el limite"


for m in [500_000, 5_000_000, 30_000_000, 80_000_000]:
    print(aprobar_credito_sin_patron(m))
print(">> Problema: un if/elif monolitico; agregar 'Director' o mover limites obliga a editar la funcion.")

Cajero aprueba $500000
Supervisor aprueba $5000000
Gerente aprueba $30000000
RECHAZADO: $80000000 excede el limite
>> Problema: un if/elif monolitico; agregar 'Director' o mover limites obliga a editar la funcion.


### Análisis del problema
- Todos los niveles de decisión están **acoplados** en una sola función.
- Agregar un nivel (p. ej. **Director**) o reordenar límites obliga a **modificar** el
  bloque: viola **Open/Closed**.
- No se puede reconfigurar la cadena en tiempo de ejecución.

## Código *con patrón* (problema resuelto)
Cada nivel es un **manejador** con `manejar(solicitud)`. Si no puede, delega en el
siguiente. La cadena se **arma** enlazando manejadores y se puede reordenar/extender.

In [2]:
from abc import ABC, abstractmethod
from typing import Optional


class Aprobador(ABC):
    def __init__(self):
        self._siguiente: Optional["Aprobador"] = None

    def enlazar(self, siguiente: "Aprobador") -> "Aprobador":
        self._siguiente = siguiente
        return siguiente  # permite encadenar: a.enlazar(b).enlazar(c)

    def manejar(self, monto: float) -> str:
        if self._puede_aprobar(monto):
            return f"{self.__class__.__name__} aprueba ${monto}"
        if self._siguiente is not None:
            return self._siguiente.manejar(monto)
        return f"RECHAZADO: ${monto} excede todos los niveles"

    @abstractmethod
    def _puede_aprobar(self, monto: float) -> bool: ...


class Cajero(Aprobador):
    def _puede_aprobar(self, monto): return monto <= 1_000_000

class Supervisor(Aprobador):
    def _puede_aprobar(self, monto): return monto <= 10_000_000

class Gerente(Aprobador):
    def _puede_aprobar(self, monto): return monto <= 50_000_000


# Se arma la cadena: cajero -> supervisor -> gerente
cajero = Cajero()
supervisor = Supervisor()
gerente = Gerente()
cajero.enlazar(supervisor).enlazar(gerente)

for m in [500_000, 5_000_000, 30_000_000, 80_000_000]:
    print(cajero.manejar(m))
print(">> Solucion: cada nivel es un manejador; agregar 'Director' es enlazar otro eslabon.")

Cajero aprueba $500000
Supervisor aprueba $5000000
Gerente aprueba $30000000
RECHAZADO: $80000000 excede todos los niveles
>> Solucion: cada nivel es un manejador; agregar 'Director' es enlazar otro eslabon.


### Verificación
- Existe una **interfaz de manejador** (`Aprobador.manejar`) y **3 manejadores concretos**
  (`Cajero`, `Supervisor`, `Gerente`).
- Cada uno **atiende o delega** al siguiente; si nadie puede, se rechaza.
- Agregar un `Director` sería crear una clase y enlazarla: sin tocar las demás.

## UML del patrón Chain of Responsibility
```plantuml
@startuml
abstract class Aprobador {
    - _siguiente : Aprobador
    + enlazar(siguiente)
    + manejar(monto)
    + _puede_aprobar(monto)
}
Aprobador <|-- Cajero
Aprobador <|-- Supervisor
Aprobador <|-- Gerente
Aprobador --> Aprobador : _siguiente
@enduml
```

## ¿Por qué Chain of Responsibility y no otro patrón?
- El problema es **quién de una secuencia de responsables atiende la solicitud**, con la
  posibilidad de delegar. Eso es exactamente la cadena de responsabilidad.
- No es Strategy: Strategy elige **un** algoritmo intercambiable; aquí la solicitud puede
  **recorrer varios** manejadores hasta encontrar el adecuado.
- No es un simple `if/elif`: la cadena se **arma dinámicamente** y cada eslabón es
  independiente, respetando **Open/Closed**.